# Exercise 2: Cost Tracking for Token-Based Inference

Cloud inference platforms like **Nebius Token Factory** charge per token, not per GPU hour.
Beyond is building a similar service - you call an OpenAI-compatible API,
and pay based on token consumption.

The cost formula:

```
cost = (prompt_tokens × input_price) + (completion_tokens × output_price)
```

In this exercise we'll:
1. See how vLLM already returns token usage in every response
2. Build a cost tracker that hooks into real workflow executions
3. Analyze per-agent costs from Exercise 1's Phoenix traces
4. Identify where the money goes in a multi-agent workflow

In [ ]:
import os
import time
from collections import defaultdict
from dataclasses import dataclass, field
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../../.env")

client = OpenAI(
    base_url=os.getenv("VLLM_BASE_URL"),
    api_key=os.getenv("VLLM_API_KEY"),
)
MODEL = os.getenv("VLLM_MODEL_NAME")

## Step 1: Token Usage is Already in Every Response

vLLM (and any OpenAI-compatible API) returns a `usage` object with every response.
This is the same data Token Factory uses for billing.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Jaka jest stolica Polski?"}],
    temperature=0.7,
    max_tokens=128,
)

print(f"Answer: {response.choices[0].message.content}")
print(f"")
print(f"--- Token Usage (from response.usage) ---")
print(f"Prompt tokens:     {response.usage.prompt_tokens:>6}")
print(f"Completion tokens: {response.usage.completion_tokens:>6}")
print(f"Total tokens:      {response.usage.total_tokens:>6}")
print(f"")
print("This is what a Token Factory bills on.")

## Step 2: Define a Pricing Model

Token Factory-style pricing has separate rates for input and output tokens.
Output tokens cost more because they're generated sequentially (slower).

Example rates (per 1M tokens) for reference:

| Model | Input | Output |
|-------|-------|--------|
| Llama 3.3 70B (Nebius) | ~$0.20 | ~$0.35 |
| GPT-4o (OpenAI) | $2.50 | $10.00 |
| Bielik-7B (Beyond) | TBD | TBD |

We'll use placeholder rates - Beyond will set the real ones.

In [ ]:
@dataclass
class TokenPricing:
    """Token Factory-style per-token pricing."""
    input_per_1m: float   # USD per 1M input tokens
    output_per_1m: float  # USD per 1M output tokens
    name: str = "default"

    def cost(self, prompt_tokens: int, completion_tokens: int) -> float:
        return (
            prompt_tokens * self.input_per_1m / 1_000_000
            + completion_tokens * self.output_per_1m / 1_000_000
        )


# Placeholder pricing for Bielik-7B on Beyond's Token Factory
BIELIK_PRICING = TokenPricing(
    name="Bielik-Minitron-7B (Beyond)",
    input_per_1m=0.10,   # $0.10 per 1M input tokens
    output_per_1m=0.20,  # $0.20 per 1M output tokens
)

# Quick demo
demo_cost = BIELIK_PRICING.cost(
    response.usage.prompt_tokens,
    response.usage.completion_tokens,
)
print(f"Pricing: {BIELIK_PRICING.name}")
print(f"  Input:  ${BIELIK_PRICING.input_per_1m:.2f} / 1M tokens")
print(f"  Output: ${BIELIK_PRICING.output_per_1m:.2f} / 1M tokens")
print(f"")
print(f"Cost of previous call: ${demo_cost:.6f}")
print(f"  ({response.usage.prompt_tokens} input + {response.usage.completion_tokens} output tokens)")

## Step 3: Build a Cost Tracker

This tracker records every LLM call with its token usage and computes costs.
In production, you'd hook this into your API gateway or use the platform's billing API.

In [ ]:
@dataclass
class CallRecord:
    """One LLM API call."""
    agent: str
    prompt_tokens: int
    completion_tokens: int
    latency_ms: float

    @property
    def total_tokens(self) -> int:
        return self.prompt_tokens + self.completion_tokens


@dataclass
class CostTracker:
    """Tracks token usage and costs across a workflow."""
    pricing: TokenPricing
    records: list = field(default_factory=list)

    def record(self, agent: str, response, latency_ms: float):
        """Record a call from an OpenAI-compatible response object."""
        self.records.append(CallRecord(
            agent=agent,
            prompt_tokens=response.usage.prompt_tokens,
            completion_tokens=response.usage.completion_tokens,
            latency_ms=latency_ms,
        ))

    def report(self):
        """Print a cost breakdown by agent."""
        by_agent = defaultdict(lambda: {"calls": 0, "input": 0, "output": 0, "latency": 0.0})
        for r in self.records:
            a = by_agent[r.agent]
            a["calls"] += 1
            a["input"] += r.prompt_tokens
            a["output"] += r.completion_tokens
            a["latency"] += r.latency_ms

        total_input = sum(r.prompt_tokens for r in self.records)
        total_output = sum(r.completion_tokens for r in self.records)
        total_cost = self.pricing.cost(total_input, total_output)

        print(f"{'='*75}")
        print(f"COST REPORT  |  Pricing: {self.pricing.name}")
        print(f"Input: ${self.pricing.input_per_1m:.2f}/1M tokens  |  Output: ${self.pricing.output_per_1m:.2f}/1M tokens")
        print(f"{'='*75}")
        print(f"{'Agent':<22} {'Calls':>5} {'Input':>8} {'Output':>8} {'Cost':>10} {'Latency':>9}")
        print(f"{'-'*75}")

        for name, d in sorted(by_agent.items(), key=lambda x: -self.pricing.cost(x[1]['input'], x[1]['output'])):
            cost = self.pricing.cost(d['input'], d['output'])
            pct = (cost / total_cost * 100) if total_cost > 0 else 0
            print(f"{name:<22} {d['calls']:>5} {d['input']:>8} {d['output']:>8} ${cost:>8.6f} {d['latency']:>7.0f}ms")

        print(f"{'-'*75}")
        print(f"{'TOTAL':<22} {len(self.records):>5} {total_input:>8} {total_output:>8} ${total_cost:>8.6f} {sum(r.latency_ms for r in self.records):>7.0f}ms")
        print(f"{'='*75}")

        # Cost split
        input_cost = self.pricing.cost(total_input, 0)
        output_cost = self.pricing.cost(0, total_output)
        print(f"\nCost split: input ${input_cost:.6f} ({input_cost/total_cost*100:.0f}%) | output ${output_cost:.6f} ({output_cost/total_cost*100:.0f}%)")

        # Per-request average
        if self.records:
            print(f"Average cost per call: ${total_cost / len(self.records):.6f}")
            print(f"Average latency:      {sum(r.latency_ms for r in self.records) / len(self.records):.0f}ms")


tracker = CostTracker(pricing=BIELIK_PRICING)
print("Cost tracker ready.")

## Step 4: Track a Simulated Multi-Agent Workflow

We'll simulate the workflow from Part 2 - an orchestrator calling a research agent
and a writing crew - making real LLM calls and tracking every one.

In [ ]:
def tracked_call(agent: str, messages: list, **kwargs) -> str:
    """Make an LLM call and record usage."""
    start = time.time()
    resp = client.chat.completions.create(model=MODEL, messages=messages, **kwargs)
    tracker.record(agent, resp, (time.time() - start) * 1000)
    return resp.choices[0].message.content


# --- Orchestrator decides what to do ---
print("[orchestrator] Routing query...")
tracked_call("orchestrator", [
    {"role": "system", "content": "Jesteś orkiestratorem. Zdecyduj, którego agenta wywołać."},
    {"role": "user", "content": "Zbadaj Tatry i napisz raport podróżniczy."},
], temperature=0.3, max_tokens=256)

# --- Research agent: multiple reasoning steps ---
print("[research_agent] Searching Wikipedia...")
tracked_call("research_agent", [
    {"role": "system", "content": "Jesteś agentem badawczym. Podaj szczegółowe fakty na temat."},
    {"role": "user", "content": "Zbadaj Tatry: geografia, dzika przyroda, statystyki turystyczne."},
], temperature=0.7, max_tokens=1024)

print("[research_agent] Follow-up search...")
tracked_call("research_agent", [
    {"role": "system", "content": "Jesteś agentem badawczym. Podaj dodatkowe szczegóły."},
    {"role": "user", "content": "Jakie szlaki turystyczne są najpopularniejsze w Tatrach? Jaka jest najlepsza pora na wizytę?"},
], temperature=0.7, max_tokens=1024)

# --- Orchestrator routes to writing crew ---
print("[orchestrator] Routing to writing crew...")
tracked_call("orchestrator", [
    {"role": "system", "content": "Przekaż wyniki badań do zespołu redakcyjnego."},
    {"role": "user", "content": "Badania zakończone. Przekaż wyniki do writing_crew w celu napisania raportu."},
], temperature=0.3, max_tokens=128)

# --- Writing crew: analyst ---
print("[crew/analyst] Analyzing findings...")
tracked_call("crew/analyst", [
    {"role": "system", "content": "Jesteś analitykiem. Wyodrębnij kluczowe wnioski z dostarczonego materiału."},
    {"role": "user", "content": "Tatry są najwyższym pasmem Karpat, sięgając 2499 m n.p.m. na Rysach. Zamieszkuje je unikalna fauna, w tym kozice i świstaki. Rocznie odwiedza je ponad 3 miliony turystów."},
], temperature=0.5, max_tokens=512)

# --- Writing crew: writer ---
print("[crew/writer] Writing report...")
tracked_call("crew/writer", [
    {"role": "system", "content": "Jesteś redaktorem podróżniczym. Napisz uporządkowany raport zawierający: podsumowanie, najważniejsze atrakcje, praktyczne wskazówki i wnioski."},
    {"role": "user", "content": "Napisz raport podróżniczy o Tatrach. Kluczowe fakty: najwyższe pasmo Karpat (2499m Rysy), unikalna fauna alpejska (kozice, świstaki, niedźwiedzie), ponad 3 mln turystów rocznie, najlepsze szlaki Morskie Oko i Orla Perć, najlepsza pora czerwiec-wrzesień, dzielone między Polskę a Słowację."},
], temperature=0.7, max_tokens=1024)

# --- Orchestrator: final answer ---
print("[orchestrator] Summarizing...")
tracked_call("orchestrator", [
    {"role": "system", "content": "Podaj krótkie podsumowanie potwierdzające ukończenie raportu."},
    {"role": "user", "content": "Raport podróżniczy o Tatrach został ukończony przez zespół redakcyjny."},
], temperature=0.3, max_tokens=128)

print("\nWorkflow complete!\n")
tracker.report()

## Step 5: Visualize Where the Money Goes

In [ ]:
# Cost by agent role
by_agent = defaultdict(lambda: {"input": 0, "output": 0})
for r in tracker.records:
    by_agent[r.agent]["input"] += r.prompt_tokens
    by_agent[r.agent]["output"] += r.completion_tokens

costs = {name: BIELIK_PRICING.cost(d["input"], d["output"]) for name, d in by_agent.items()}
max_cost = max(costs.values())
total = sum(costs.values())

print("Cost by Agent")
print("=" * 65)
for name, cost in sorted(costs.items(), key=lambda x: -x[1]):
    bar_len = int((cost / max_cost) * 30)
    pct = cost / total * 100
    print(f"{name:<22} {'█' * bar_len:<30} ${cost:.6f} ({pct:.0f}%)")

print(f"\n{'Total':<22} {'':30} ${total:.6f}")

# Cost by role (orchestration vs research vs writing)
print(f"\n\nCost by Role")
print("=" * 65)
roles = defaultdict(float)
for name, cost in costs.items():
    if "orchestrator" in name:
        roles["Orchestration overhead"] += cost
    elif "research" in name:
        roles["Research (LangChain)"] += cost
    elif "crew" in name:
        roles["Writing (CrewAI)"] += cost

for role, cost in sorted(roles.items(), key=lambda x: -x[1]):
    pct = cost / total * 100
    print(f"  {role:<30} ${cost:.6f} ({pct:.0f}%)")

## Step 6: Compare Pricing Scenarios

How much would this workflow cost on different platforms?

In [ ]:
scenarios = [
    TokenPricing(name="Beyond Token Factory (Bielik-7B)", input_per_1m=0.10, output_per_1m=0.20),
    TokenPricing(name="Nebius (Llama-3.3-70B)",          input_per_1m=0.20, output_per_1m=0.35),
    TokenPricing(name="OpenAI (GPT-4o-mini)",            input_per_1m=0.15, output_per_1m=0.60),
    TokenPricing(name="OpenAI (GPT-4o)",                 input_per_1m=2.50, output_per_1m=10.00),
]

total_input = sum(r.prompt_tokens for r in tracker.records)
total_output = sum(r.completion_tokens for r in tracker.records)

print(f"Workflow tokens: {total_input} input + {total_output} output = {total_input + total_output} total")
print(f"Calls: {len(tracker.records)}")
print()
print(f"{'Platform':<38} {'Cost':>10} {'vs Beyond':>10}")
print("-" * 60)

base_cost = scenarios[0].cost(total_input, total_output)
for s in scenarios:
    cost = s.cost(total_input, total_output)
    multiplier = cost / base_cost if base_cost > 0 else 0
    print(f"{s.name:<38} ${cost:>8.6f} {multiplier:>9.1f}x")

## Step 7: Estimate Monthly Cost at Scale

What would this workflow cost if run 1,000 times per day?

In [ ]:
requests_per_day = 1000
cost_per_request = BIELIK_PRICING.cost(total_input, total_output)

daily = cost_per_request * requests_per_day
monthly = daily * 30

print(f"Per request:  ${cost_per_request:.6f}")
print(f"Daily ({requests_per_day:,} req): ${daily:.2f}")
print(f"Monthly:      ${monthly:.2f}")
print()
print(f"Token breakdown per request:")
print(f"  Input:  {total_input:>6} tokens (${BIELIK_PRICING.cost(total_input, 0):.6f})")
print(f"  Output: {total_output:>6} tokens (${BIELIK_PRICING.cost(0, total_output):.6f})")
print()

# Which agent to optimize first?
print("Cost reduction opportunities:")
for name, cost in sorted(costs.items(), key=lambda x: -x[1])[:3]:
    monthly_agent = cost * requests_per_day * 30
    saving_20pct = monthly_agent * 0.2
    print(f"  {name}: ${monthly_agent:.2f}/mo -> 20% optimization saves ${saving_20pct:.2f}/mo")

## Step 8: Analyze Phoenix Traces for Real Workflow Costs

If you ran Exercise 1 (observability), Phoenix captured token counts for every
real `nat run` call. Let's query those traces.

In [ ]:
try:
    import phoenix as px
    phoenix_client = px.Client()
    traces_df = phoenix_client.get_spans_dataframe()

    if traces_df is not None and len(traces_df) > 0:
        # Filter for LLM spans that have token usage
        llm_spans = traces_df[
            traces_df.columns.intersection(["name", "llm.token_count.prompt", "llm.token_count.completion", "latency_ms"])
        ].dropna(subset=[c for c in ["llm.token_count.prompt"] if c in traces_df.columns])

        if len(llm_spans) > 0:
            total_prompt = int(llm_spans["llm.token_count.prompt"].sum())
            total_completion = int(llm_spans["llm.token_count.completion"].sum())
            real_cost = BIELIK_PRICING.cost(total_prompt, total_completion)

            print(f"Phoenix traces: {len(llm_spans)} LLM calls captured")
            print(f"Total tokens:   {total_prompt} input + {total_completion} output")
            print(f"Real cost:      ${real_cost:.6f}")
        else:
            print("No LLM spans with token data found. Run Exercise 1 first.")
    else:
        print("No traces found. Run Exercise 1 first.")

except Exception as e:
    print(f"Phoenix not available: {e}")
    print("Run Exercise 1 first to capture traces, then come back here.")

## Key Takeaways

1. **Token Factory pricing is simple:** `(input_tokens × rate) + (output_tokens × rate)`
2. **vLLM already returns usage data** in every response - no extra instrumentation needed
3. **Output tokens cost more** (and take more time) - optimizing output length has double impact
4. **Orchestration overhead is real** - the orchestrator agent consumes tokens just to decide what to do
5. **Multi-agent workflows multiply costs** - each agent makes its own LLM calls

**For production:** Hook cost tracking into your API gateway or use the platform's
billing API. Set per-user/per-workflow budgets. Alert on cost spikes.

**Next:** Exercise 3 - Optimize parameters to reduce costs while maintaining quality